# 2 — Retrieval and Generation

Now we ask questions of the index we built in notebook 1.

Four things happen, and all four are used in real systems:

```
search  →  rerank  →  filter  →  fetch table rows  →  answer
```

The code lives in the `rag` package, same as notebook 1. Each cell here is a few
lines — enough to see what a step does. Open `rag/retrieval.py` if you want the
details.

| § | What |
|---|---|
| 0 | Setup |
| 1 | Search and rerank |
| 2 | Filtering with metadata |
| 3 | Table rows |
| 4 | Build the prompt and answer |

---
## 0. Setup

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os

os.environ.setdefault("OPENAI_API_KEY", "")
os.environ.setdefault("PINECONE_API_KEY", "")

''

In [5]:
from dotenv import load_dotenv

# Read environment variables from the .env file, if one exists.
load_dotenv()

True

In [10]:
from dotenv import load_dotenv
from pathlib import Path
import os
from dotenv import load_dotenv
from pathlib import Path
import os

env_file = Path(
    r"C:\GenAI\Training\genai-agenticai-course\module-5\rag\.env"
)

print("Env file exists:", env_file.exists())

load_dotenv(env_file, override=True)

print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Pinecone key loaded:", bool(os.getenv("PINECONE_API_KEY")))
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Pinecone key loaded:", bool(os.getenv("PINECONE_API_KEY")))

Env file exists: True
OpenAI key loaded: True
Pinecone key loaded: True
OpenAI key loaded: True
Pinecone key loaded: True


In [12]:
from rag.index import open_index
from rag.retrieval import search

index = open_index()

query = "What kind of leaves employess in Goa and India are entitled to?"

results = search(
    index,
    query,
    top_k=5
)

print("Number of results:", len(results))

for i, result in enumerate(results, 1):
    print("\n" + "=" * 80)
    print(f"RESULT {i}")
    print("=" * 80)

    print("Chunk ID :", result["chunk_id"])
    print("Page     :", result["page"])
    print("Type     :", result["content_type"])
    print("Dense    :", result["dense"])
    print("Rerank   :", result["rerank"])
    print("\nText:")
    print(result["text"][:1000])

embeddings created ==============
Number of results: 5

RESULT 1
Chunk ID : leavepolicy:35b04612ef2bf3a2:0
Page     : 1
Type     : text
Dense    : 0.7171
Rerank   : 0.9979

Text:
General Overview of Leave policy :
Employees, across all industries in Goa, India, are entitled to a certain number of leaves per year aside from the holidays and days off. In units covered under The Factories Act, three types of leaves are generally follo ed namely earned leave, sick leave and casual leave hich an employee can avail ithout loss of pay.
-  Casual leave.
-  Sick leave
-  Privilege leave
All regular employees are entitled to leave as per the standard leave policy.
Commencement of Leave Period is calendar year i.e. 1st January to 31st December every
year. Employee need to apply for each leave and take approval except in cases here approval could not be taken in advance usually for casual or sick leaves. Grant of leave shall depend upon the policies of the orkplace and is at the discretion of t

Two score columns. `dense` is the vector distance. `rerank` is the cross-encoder
reading your question and the chunk together.

Where they disagree is where the second stage earned its cost.